<a href="https://colab.research.google.com/github/CrUz-035/Prediccion-de-compra-de-casas/blob/main/Predicci%C3%B3n_de_compra_de_casas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import image_dataset_from_directory
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
#Se instala la libreria donde se encuentra CelebA
!pip install optuna #instalamos optuna
import optuna
!pip install -q mlflow #Instalamos Mlflow
import mlflow
! pip install jupyterlab jupyterlab-optuna
mlflow.tensorflow.autolog()
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Input
#Importamos las funciones a ocupar en el entrenamiento.
from tensorflow.keras.optimizers import RMSprop, SGD, Adam
from keras.callbacks import ModelCheckpoint, EarlyStopping

from tensorflow.keras.layers import Input

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

from sklearn.model_selection import train_test_split

!pip install -U kaleido


In [ ]:
#Acceso a Dagshub/mlflowd
REPO_NAME = "Prediccion_venta_casas"
REPO_OWNER = "CrUz-035"
USER_NAME = "CrUz-035"

!pip install -q mlflow dagshub
import mlflow
from getpass import getpass

mlflow.tensorflow.autolog()

import dagshub
import os
from getpass import getpass

#llaves de acceso
from google.colab import userdata

os.environ['MLFLOW_TRACKING_USERNAME'] = USER_NAME
os.environ['MLFLOW_TRACKING_PASSWORD'] =userdata.get('TOKEN')

mlflow.set_tracking_uri(f'https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow')
mlflow.set_experiment("Prentrenado")

# Nombre del experimento
mlflow.set_experiment("Optuna_casa")

<Experiment: artifact_location='mlflow-artifacts:/6562a59cbdfa4ac3a802dea17fdd9e61', creation_time=1777125594354, experiment_id='1', last_update_time=1777125594354, lifecycle_stage='active', name='Optuna_casa', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
def objective(trial):
    with mlflow.start_run(nested=True):

        model = Sequential()

        n_layers = trial.suggest_int("n_layers", 1, 5) #número de capas a probar

        for i in range(n_layers):

            n_units = trial.suggest_categorical(
                f"n_units_l{i}", [32, 64, 128, 256, 512] #Neuronas
            )

            activation = trial.suggest_categorical(  #Función activación
                f"activation_l{i}", ["relu", "tanh", "sigmoid"]
            )

            if i == 0:
                model.add(Input(shape=(X_train.shape[1],)))
                model.add(Dense(n_units, activation=activation))
            else:
                model.add(Dense(n_units, activation=activation))

        model.add(Dense(1, activation="sigmoid"))#Clasificación final binaria

        optimizer_name = trial.suggest_categorical( #Optimizador
            "optimizer", ["Adam", "SGD", "RMSprop"]
        )

        lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True) #Learning rate

        if optimizer_name == "Adam":
            optimizer = Adam(learning_rate=lr)
        elif optimizer_name == "SGD":
            optimizer = SGD(learning_rate=lr)
        else:
            optimizer = RMSprop(learning_rate=lr)

        model.compile(
            loss="binary_crossentropy",
            optimizer=optimizer,
            metrics=["accuracy", "precision", "recall", "auc"]
        )

        #Entrenamiento
        model.fit(
          X_train, y_train,
          epochs=15,
          batch_size=128,
          verbose=0
        )

        #Prueba
        results = model.evaluate(
          X_test, y_test,
          verbose=0
        )

        test_loss = results[0]
        test_accuracy = results[1]
        test_precision = results[2]
        test_recall = results[3]
        test_auc = results[4]

        # Guardar en MLflow (todas las métricas)
        mlflow.log_metrics({
            "test_accuracy": test_accuracy,
            "test_precision": test_precision,
            "test_recall": test_recall,
            "test_auc": test_auc,
            "test_loss": test_loss
        })

        #Registro en dagshub
        mlflow.log_params({
            "n_layers": n_layers,
            "optimizer": optimizer_name,
            "lr": lr
        })

        # Guardar cada capa
        for i in range(n_layers):
            mlflow.log_param(f"n_units_l{i}", trial.params[f"n_units_l{i}"])
            mlflow.log_param(f"activation_l{i}", trial.params[f"activation_l{i}"])

        mlflow.log_metric("test_accuracy", test_accuracy) # Use test_accuracy here

        # guardar modelo
        mlflow.keras.log_model(model, name="model")

        return test_accuracy # Return test_accuracy as it's the parameter to optimize

In [ ]:
df = pd.read_csv("/content/global_house_purchase_dataset.csv") #Cargar dataset

print(df.head())
print(df.columns)

   property_id       country          city property_type furnishing_status  \
0            1        France     Marseille     Farmhouse    Semi-Furnished   
1            2  South Africa     Cape Town     Apartment    Semi-Furnished   
2            3  South Africa  Johannesburg     Farmhouse    Semi-Furnished   
3            4       Germany     Frankfurt     Farmhouse    Semi-Furnished   
4            5  South Africa  Johannesburg     Townhouse   Fully-Furnished   

   property_size_sqft    price  constructed_year  previous_owners  rooms  ...  \
0                 991   412935              1989                6      6  ...   
1                1244   224538              1990                4      8  ...   
2                4152   745104              2019                5      2  ...   
3                3714  1110959              2008                1      3  ...   
4                 531    99041              2007                6      3  ...   

   customer_salary  loan_amount  loan_tenure

In [ ]:
X = df.drop("decision", axis=1) #Se separa decisión de los demás datos.
y = df["decision"]
X = pd.get_dummies(X)

In [ ]:
from sklearn.preprocessing import StandardScaler

# 70% Entrenamiento, 30% se separa después
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
# dividir el 30% en 20% Prueba y 10% Validación
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=1/3, random_state=42
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train) #Se reescalan datos para poder entrenar.
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)


X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)#Se convierten datos a numéricos
y_test = y_test.astype(np.float32)
y_val = y_val.astype(np.float32) #Solo se convierte pero no se utiliza

In [ ]:
study = optuna.create_study(direction="maximize") #Se entrena con optuna
study.optimize(objective, n_trials=50)

import matplotlib.pyplot as plt
from optuna.visualization import plot_optimization_history, plot_param_importances
from optuna.visualization import plot_parallel_coordinate, plot_contour
from optuna.visualization import plot_slice, plot_edf

fig1 = plot_optimization_history(study)
fig1.show()

# 2. Importancia de los hiperparámetros
fig2 = plot_param_importances(study)
fig2.show()

# 3. Coordenadas paralelas (relación entre parámetros y resultados)
fig3 = plot_parallel_coordinate(study)
fig3.show()

# 4. Gráfica de contorno (interacción entre 2 parámetros)
fig4 = plot_contour(study, params=['n_units_l0', 'lr'])
fig4.show()

# 5. Gráfica de slice (distribución de resultados por parámetro)
fig5 = plot_slice(study)
fig5.show()

[I 2026-04-25 15:26:26,981] A new study created in memory with name: no-name-e6ae372a-8646-4f04-a967-30408582d032
2026/04/25 15:26:33 WARNING mlflow.tensorflow: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


2026/04/25 15:26:49 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not available.
2026/04/25 15:26:54 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not available.
2026/04/25 15:26:58 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not available.
2026/04/25 15:27:01 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not availab

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 535ms/step


2026/04/25 15:27:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 15:28:37 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


🏃 View run clean-cow-537 at: https://dagshub.com/CrUz-035/Prediccion_venta_casas.mlflow/#/experiments/1/runs/3d1ecc53e2484e6ab09dc06c6cb9caa0
🧪 View experiment at: https://dagshub.com/CrUz-035/Prediccion_venta_casas.mlflow/#/experiments/1


...

2026/04/25 17:01:25 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not available.
2026/04/25 17:01:29 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not available.
2026/04/25 17:01:33 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not available.
2026/04/25 17:01:37 WARNING mlflow.utils.checkpoint_utils: Checkpoint logging is skipped, because checkpoint 'save_best_only' config is True, it requires to compare the monitored metric value, but the provided monitored metric value is not availab

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step


2026/04/25 17:02:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 17:02:51 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


🏃 View run funny-auk-609 at: https://dagshub.com/CrUz-035/Prediccion_venta_casas.mlflow/#/experiments/1/runs/0176615f1f184a7d8a2ae13746d1aa8e
🧪 View experiment at: https://dagshub.com/CrUz-035/Prediccion_venta_casas.mlflow/#/experiments/1


[I 2026-04-25 17:03:06,088] Trial 49 finished with value: 0.9964500069618225 and parameters: {'n_layers': 1, 'n_units_l0': 256, 'activation_l0': 'sigmoid', 'optimizer': 'Adam', 'lr': 0.006330246843851688}. Best is trial 42 with value: 0.9986249804496765.
/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [ ]:
best_params = study.best_params
print(best_params) #Se utilizan los mejores parámetros obtenidos por Optuna

{'n_layers': 1, 'n_units_l0': 64, 'activation_l0': 'sigmoid', 'optimizer': 'Adam', 'lr': 0.0007463336993080731}


In [ ]:
def build_best_model(params): #Modelo final a entrenar

    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],))) #Número de columnas del dataset
    for i in range(params["n_layers"]): #Capas propuestas por optuna
        model.add(Dense(
            params[f"n_units_l{i}"],
            activation=params[f"activation_l{i}"] #Activación de optuna
        ))
    model.add(Dense(1, activation="sigmoid")) #Capa de salida.
    # Optimizador a usar
    if params["optimizer"] == "Adam":
        optimizer = Adam(learning_rate=params["lr"])
    elif params["optimizer"] == "SGD":
        optimizer = SGD(learning_rate=params["lr"])
    else:
        optimizer = RMSprop(learning_rate=params["lr"])

    model.compile( #Clasificación binaria
        loss="binary_crossentropy",
        optimizer=optimizer,
        metrics=["accuracy"]
    )
    return model

In [ ]:
model = build_best_model(best_params)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128,
    verbose=1
)

with mlflow.start_run(run_name="FINAL_MODEL"):

    # Evaluar en validación
    val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)

    # Guardar parámetros
    mlflow.log_params(best_params)

    # Guardar métricas
    mlflow.log_metrics({
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "train_loss": history.history['loss'][-1],  # última época
        "train_accuracy": history.history.get('accuracy', [0])[-1]
    })

    # Guardar modelo
    mlflow.keras.log_model(model, name="final_model")

2026/04/25 17:03:09 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'cb9278db72ee424a9fd224805cb6311e', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current tensorflow workflow
2026/04/25 17:03:10 WARNING mlflow.tensorflow: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


Epoch 1/50
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7921 - loss: 0.4142

1094/1094 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.8747 - loss: 0.2920 - val_accuracy: 0.9405 - val_loss: 0.1627
Epoch 2/50
1081/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9453 - loss: 0.1445

1094/1094 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9463 - loss: 0.1366 - val_accuracy: 0.9496 - val_loss: 0.1241
Epoch 3/50
1087/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9525 - loss: 0.1172

1094/1094 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9546 - loss: 0.1119 - val_accuracy: 0.9601 - val_loss: 0.1017
Epoch 4/50
1081/1094 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9632 - loss: 0.0958

...

1094/1094 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9998 - loss: 0.0012 - val_accuracy: 0.9990 - val_loss: 0.0024
Epoch 42/50
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9998 - loss: 0.0012 - val_accuracy: 0.9990 - val_loss: 0.0026
Epoch 43/50
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9998 - loss: 0.0011 - val_accuracy: 0.9990 - val_loss: 0.0024
Epoch 44/50
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9999 - loss: 0.0010 - val_accuracy: 0.9990 - val_loss: 0.0025
Epoch 45/50
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9999 - loss: 9.9888e-04 - val_accuracy: 0.9989 - val_loss: 0.0026
Epoch 46/50
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9999 - loss: 9.1444e-04 - val_accuracy: 0.9991 - val_loss: 0.0026
Epoch 47/50
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 1.0000 - loss: 8.6332e-04 - val_accuracy: 0.9987 - val_loss: 0.0029
Epoch 48/50
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9999 - loss: 8

2026/04/25 17:06:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run masked-rook-870 at: https://dagshub.com/CrUz-035/Prediccion_venta_casas.mlflow/#/experiments/1/runs/cb9278db72ee424a9fd224805cb6311e
🧪 View experiment at: https://dagshub.com/CrUz-035/Prediccion_venta_casas.mlflow/#/experiments/1


2026/04/25 17:07:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during tensorflow autologging: BAD_REQUEST: Response: {'error_code': 'BAD_REQUEST'}
2026/04/25 17:07:13 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.


🏃 View run FINAL_MODEL at: https://dagshub.com/CrUz-035/Prediccion_venta_casas.mlflow/#/experiments/1/runs/c78521660193473b839e9f73b542a3af
🧪 View experiment at: https://dagshub.com/CrUz-035/Prediccion_venta_casas.mlflow/#/experiments/1


In [ ]:
model.summary()

Model: "sequential_50"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_151 (Dense)               │ (None, 64)             │         5,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_152 (Dense)               │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,133 (63.02 KB)

 Trainable params: 5,377 (21.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 10,756 (42.02 KB)